# K-State Baseball — Arm Care Command Center

This notebook runs the Command Center Streamlit app (`app.py`) and lets you check the data from Python.
Run the cells top to bottom. **Step 3** starts the app at http://localhost:8501, and **Step 5** stops it.

Everything entered in the app saves to `data/command_center.db` automatically.

## 1 · Install packages (first time only)

In [ ]:
import sys, subprocess
subprocess.call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("Packages ready.")

## 2 · Load the data and check it
On the very first run this also builds the database from `data/seed_command_center.json` and the ACWR workbook.

In [ ]:
import os
from datetime import date
import pandas as pd
import storage as db, model as m, assess as ax

db.seed_if_empty(os.path.join("data", "ACWR Throwing Plan Fall 2026.xlsx"),
                 os.path.join("data", "seed_command_center.json"))
players = db.load_players()
print(f"{len(players)} athletes · {(players.grp == 'pitcher').sum()} pitchers · "
      f"{len(db.load_force())} force-plate tests · {len(db.load_library())} library exercises")
players[["name", "number", "position", "grp", "year", "bt", "avg_top_velo"]]

In [ ]:
# Staff throwing status as of today (same math as the app's Team ▸ Throwing view)
today = date.today()
rows = []
for p in players[players.grp == "pitcher"].to_dict("records"):
    avg = None if pd.isna(p["avg_top_velo"]) else p["avg_top_velo"]
    log = db.load_log(p["id"])
    calc = [m.calc_row(r, avg) for r in log.to_dict("records")]
    daily = m.daily_totals(pd.DataFrame({"date": pd.to_datetime(log["date"]).dt.date,
                                         "set_workload": [c["set_workload"] for c in calc]}) if len(log) else None)
    s = m.status(daily, today)
    rows.append({"Player": p["name"], "Sessions": len(log), "Acute": round(s["acute"], 1),
                 "Chronic": round(s["chronic"], 1), "ACWR": None if s["acwr"] is None else round(s["acwr"], 2), "Zone": s["zone"]})
pd.DataFrame(rows)

In [ ]:
# Assessment flags across the team
flags = []
for pid, a in db.all_assess().items():
    for f in ax.compute_flags(a):
        flags.append({"Athlete": players.set_index("id").loc[pid, "name"], "Region": f["cat"], "Level": f["level"], "Flag": f["label"]})
pd.DataFrame(flags).sort_values(["Level", "Region", "Athlete"])

## 3 · Start the app
This opens the Command Center at **http://localhost:8501**. The app keeps running in the background while you use the notebook.

In [ ]:
import webbrowser, time
app = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py",
                        "--server.port", "8501", "--server.headless", "true"])
time.sleep(4)
webbrowser.open("http://localhost:8501")
print("Command Center running at http://localhost:8501  (run Step 5 to stop it)")

## 4 · (Optional) Show it inside the notebook

In [ ]:
from IPython.display import IFrame
IFrame("http://localhost:8501", width="100%", height=900)

## 5 · Stop the app

In [ ]:
app.terminate()
print("Stopped.")